In [ ]:
#| default_exp ai

# AI

> Amazon Bedrock foundation models and Amazon OpenSearch Service for GenAI workloads.

In [ ]:
#| export
import json

## Amazon Bedrock

Bedrock is serverless — foundation models require no provisioning. Just invoke.

```python
auth = AWSAuth()
print(invoke_model(auth, prompt='What is the capital of France?'))
```

In [ ]:
#| export
def _bedrock_client(auth):
    return auth.session.client('bedrock')

def _bedrock_runtime(auth):
    return auth.session.client('bedrock-runtime')

def _bedrock_agent_client(auth):
    return auth.session.client('bedrock-agent')

def list_bedrock_models(auth) -> list:
    'List available Bedrock foundation models (on-demand access).'
    return _bedrock_client(auth).list_foundation_models(
        byInferenceType='ON_DEMAND')['modelSummaries']

def invoke_model(auth, prompt, model_id='anthropic.claude-3-5-sonnet-20241022-v2:0',
                 max_tokens=1024, **_) -> str:
    'Invoke a Bedrock foundation model. Returns text response.'
    body = json.dumps({
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': max_tokens,
        'messages': [{'role': 'user', 'content': prompt}],
    })
    resp = _bedrock_runtime(auth).invoke_model(
        modelId=model_id, body=body, contentType='application/json')
    return json.loads(resp['body'].read())['content'][0]['text']

## Bedrock Knowledge Bases

RAG pipeline: S3 data source → OpenSearch vector store → Bedrock KB.

```python
create_kb(auth, 'my-kb', bucket='my-bucket', role_arn=role['Role']['Arn'])
```

In [ ]:
#| export
def create_kb(auth, name, bucket, role_arn,
              embed_model='amazon.titan-embed-text-v2:0', **_) -> dict:
    'Create a Bedrock Knowledge Base backed by S3 + OpenSearch for RAG.'
    client = _bedrock_agent_client(auth)
    os_endpoint = f'https://{name}-search.{auth.region}.es.amazonaws.com'
    kb = client.create_knowledge_base(
        name=name,
        roleArn=role_arn,
        knowledgeBaseConfiguration={
            'type': 'VECTOR',
            'vectorKnowledgeBaseConfiguration': {
                'embeddingModelArn':
                    f'arn:aws:bedrock:{auth.region}::foundation-model/{embed_model}',
            },
        },
        storageConfiguration={
            'type': 'OPENSEARCH_SERVERLESS',
            'opensearchServerlessConfiguration': {
                'collectionArn':
                    f'arn:aws:aoss:{auth.region}:{auth.account_id}:collection/{name}-search',
                'vectorIndexName': f'{name}-index',
                'fieldMapping': {
                    'vectorField': 'embedding',
                    'textField': 'content',
                    'metadataField': 'metadata',
                },
            },
        },
    )['knowledgeBase']
    # attach S3 data source
    kb_data_source(auth, kb['knowledgeBaseId'], bucket)
    return kb

def kb_data_source(auth, kb_id, bucket, prefix='') -> dict:
    'Add an S3 data source to a Bedrock Knowledge Base.'
    s3_uri = f's3://{bucket}/{prefix}' if prefix else f's3://{bucket}/'
    return _bedrock_agent_client(auth).create_data_source(
        knowledgeBaseId=kb_id,
        name=f'{kb_id}-s3',
        dataSourceConfiguration={
            'type': 'S3',
            's3Configuration': {'bucketArn': f'arn:aws:s3:::{bucket}',
                                'inclusionPrefixes': [prefix] if prefix else []},
        },
    )['dataSource']

## Amazon OpenSearch Service

Managed vector search for RAG pipelines — equivalent to Azure AI Search.

```python
create_opensearch(auth, 'my-search', **ISO27001)
print(opensearch_endpoint(auth, 'my-search'))
```

In [ ]:
#| export
def _os_client(auth):
    return auth.session.client('opensearch')

def create_opensearch(auth, name, engine_version='OpenSearch_2.13',
                      instance_type='r6g.large.search', instance_count=1,
                      encryption=True, tags=None, **_) -> dict:
    'Create or update an OpenSearch domain with encryption-at-rest enabled.'
    client = _os_client(auth)
    tag_list = [{'Key': k, 'Value': v} for k, v in (tags or {}).items()]
    kwargs = dict(
        DomainName=name,
        EngineVersion=engine_version,
        ClusterConfig={
            'InstanceType': instance_type,
            'InstanceCount': instance_count,
        },
        EncryptionAtRestOptions={'Enabled': encryption},
        NodeToNodeEncryptionOptions={'Enabled': True},
        DomainEndpointOptions={'EnforceHTTPS': True, 'TLSSecurityPolicy': 'Policy-Min-TLS-1-2-2019-07'},
        TagList=tag_list,
    )
    try:
        return client.create_domain(**kwargs)['DomainStatus']
    except client.exceptions.ResourceAlreadyExistsException:
        return client.describe_domain(DomainName=name)['DomainStatus']

def opensearch_endpoint(auth, name) -> str:
    'Return the OpenSearch domain endpoint URL.'
    domain = _os_client(auth).describe_domain(DomainName=name)['DomainStatus']
    return f'https://{domain["Endpoint"]}'

def opensearch_admin_creds(auth, name) -> dict:
    'Return master user credentials stored in Secrets Manager for the domain.'
    from .network import get_secret
    return json.loads(get_secret(auth, f'opensearch/{name}/admin'))